# Week 4 Tutorial: Pandas Grouping, Merging, and Feature Engineering

**AI in Data Science | Suggested time: 85-90 minutes | Google Colab**

## Driving question

How can we combine scores, attendance, and study-time data to describe student progress responsibly?

## Learning goals

You will use `groupby()`, aggregation, pivot tables, `concat()`, `merge()`, `join()`, calculated columns, `transform()`, and feature engineering.

> Keep this notebook and all three CSV files in the same Colab session. Upload the CSV files through the Files panel before running the loading cell.

## 1. Why data comes in multiple tables

Schools often store assessment, attendance, and study information separately. A shared **key**, such as `student_id`, connects the tables. Before combining tables, always ask:

- Is the key unique where it should be?
- Which table should determine the rows we keep?
- Could the merge duplicate or remove records?

In [ ]:
# Import Pandas and use its standard short name.
import pandas as pd
# Read the long-format score records from the CSV file.
scores = pd.read_csv("student_scores.csv")
# Read the attendance records from the CSV file.
attendance = pd.read_csv("student_attendance.csv")
# Read the study-time records from the CSV file.
study_time = pd.read_csv("student_study_time.csv")
# Display the first five score records.
display(scores.head())
# Display the first five attendance records.
display(attendance.head())
# Display the first five study-time records.
display(study_time.head())

In [ ]:
# Print the shape of each DataFrame.
print("Scores shape:", scores.shape)
# Print the attendance shape.
print("Attendance shape:", attendance.shape)
# Print the study-time shape.
print("Study-time shape:", study_time.shape)
# Check whether attendance student IDs are unique.
print("Attendance IDs unique:", attendance["student_id"].is_unique)
# Check whether study-time student IDs are unique.
print("Study-time IDs unique:", study_time["student_id"].is_unique)
# Count unique students in the score table.
print("Students in scores:", scores["student_id"].nunique())

### Checkpoint 1

Why is `student_id` repeated in `scores` but expected to be unique in `attendance`? Predict what would happen if attendance had two rows for the same student.

## 2. Grouping: split, apply, combine

`groupby()` splits rows into groups, applies a calculation to each group, and combines the answers. Use it for questions containing the word **each** or **per**.

In [ ]:
# Group score records by subject and calculate the mean score for each subject.
subject_means = scores.groupby("subject", as_index=False)["score"].mean()
# Rename the calculated score column for clarity.
subject_means = subject_means.rename(columns={"score": "mean_score"})
# Display the subject-level means.
display(subject_means)

In [ ]:
# Group by subject and assessment, then calculate several named statistics.
score_summary = scores.groupby(["subject", "assessment"], as_index=False).agg(
    mean_score=("score", "mean"),
    median_score=("score", "median"),
    minimum_score=("score", "min"),
    maximum_score=("score", "max"),
    student_count=("student_id", "nunique"),
)
# Round the numeric summary columns to one decimal place.
score_summary[["mean_score", "median_score", "minimum_score", "maximum_score"]] = score_summary[["mean_score", "median_score", "minimum_score", "maximum_score"]].round(1)
# Display the grouped summary.
display(score_summary)

In [ ]:
# Calculate the mean score for every grade and subject combination.
grade_subject_summary = scores.groupby(["grade", "subject"], as_index=False)["score"].mean()
# Rename the summary column.
grade_subject_summary = grade_subject_summary.rename(columns={"score": "mean_score"})
# Display the two-level grouping result.
display(grade_subject_summary)

## 3. `transform()`: group result returned to every original row

Aggregation makes fewer rows. `transform()` preserves the original row count, which is useful when each row needs its group's statistic.

In [ ]:
# Calculate the mean score within each subject and assessment group for every row.
scores["group_mean_score"] = scores.groupby(["subject", "assessment"])["score"].transform("mean")
# Calculate how far each score is above or below its group mean.
scores["difference_from_group_mean"] = scores["score"] - scores["group_mean_score"]
# Display selected rows with the transformed features.
display(scores[["name", "subject", "assessment", "score", "group_mean_score", "difference_from_group_mean"]].head(8))

## 4. Pivot tables: reshape long data into a comparison grid

The score table is **long**: one measurement per row. A pivot table can make it **wide**: one row per student with separate Pre and Post columns.

In [ ]:
# Create a student-by-subject table with Pre and Post score columns.
score_pivot = scores.pivot_table(index=["student_id", "name", "grade", "club"], columns=["subject", "assessment"], values="score", aggfunc="mean")
# Flatten the two-level column labels into single readable names.
score_pivot.columns = [f"{subject.lower()}_{assessment.lower()}" for subject, assessment in score_pivot.columns]
# Move index labels back into regular columns.
score_pivot = score_pivot.reset_index()
# Display the reshaped score table.
display(score_pivot.head())

### Checkpoint 2

Why did the pivot table reduce 48 score rows to 12 student rows? When might `aggfunc="mean"` hide duplicate measurements?

## 5. `concat()`: stack tables with the same columns

Use `concat()` when tables describe the same kind of records. It stacks rows; it does not match student IDs.

In [ ]:
# Select Pre assessment rows as the first compatible table.
pre_scores = scores.loc[scores["assessment"] == "Pre", ["student_id", "name", "grade", "club", "subject", "assessment", "score"]]
# Select Post assessment rows as the second compatible table.
post_scores = scores.loc[scores["assessment"] == "Post", ["student_id", "name", "grade", "club", "subject", "assessment", "score"]]
# Stack the two tables and create fresh row labels.
recombined_scores = pd.concat([pre_scores, post_scores], ignore_index=True)
# Print the shapes before and after concatenation.
print("Pre:", pre_scores.shape, "Post:", post_scores.shape, "Combined:", recombined_scores.shape)
# Display the first five combined rows.
display(recombined_scores.head())

## 6. `merge()`: match rows by a shared key

- **inner**: only keys found in both tables
- **left**: every key from the left table
- **outer**: every key from either table

Attendance intentionally includes `S13`, who has no score record. This makes join behavior visible.

In [ ]:
# Make a small student list from the reshaped score table.
score_students = score_pivot[["student_id", "name"]]
# Perform an inner merge and keep only IDs found in both tables.
inner_demo = score_students.merge(attendance, on="student_id", how="inner")
# Perform a left merge and keep every student from the score table.
left_demo = score_students.merge(attendance, on="student_id", how="left")
# Perform an outer merge and label where each row came from.
outer_demo = score_students.merge(attendance, on="student_id", how="outer", indicator=True)
# Print each merge result's number of rows.
print("Inner rows:", len(inner_demo), "Left rows:", len(left_demo), "Outer rows:", len(outer_demo))
# Display the rows that were not matched in both tables.
display(outer_demo.loc[outer_demo["_merge"] != "both"])

In [ ]:
# Merge student scores with attendance while checking for many-to-one matching.
student_analysis = score_pivot.merge(attendance, on="student_id", how="left", validate="one_to_one")
# Merge the result with study time and keep every score student.
student_analysis = student_analysis.merge(study_time, on="student_id", how="left", validate="one_to_one")
# Display the combined student-level table.
display(student_analysis.head())
# Count missing values after both merges.
display(student_analysis.isna().sum())

### Why `validate` matters

`validate="one_to_one"` turns an unexpected duplicate key into an error instead of silently multiplying rows. This is one of the most important safety habits in real data analysis.

## 7. `join()`: merge by index

`join()` is convenient after the key has been made the index. It solves the same matching problem using row labels.

In [ ]:
# Set student_id as the index of the score table.
score_indexed = score_pivot.set_index("student_id")
# Set student_id as the index of the study-time table.
study_indexed = study_time.set_index("student_id")
# Join study-time columns to score rows by matching index labels.
joined_demo = score_indexed.join(study_indexed, how="left", validate="one_to_one")
# Display the first five joined rows.
display(joined_demo.head())

## 8. Feature engineering: turn raw fields into useful measurements

A **feature** is a measurable input used for analysis or machine learning. A good feature should have a clear meaning, correct units, and no information from the future.

We will create improvement, attendance rate, study-session length, overall post score, and a transparent support flag.

In [ ]:
# Calculate improvement in Math.
student_analysis["math_improvement"] = student_analysis["math_post"] - student_analysis["math_pre"]
# Calculate improvement in Science.
student_analysis["science_improvement"] = student_analysis["science_post"] - student_analysis["science_pre"]
# Calculate average improvement across both subjects.
student_analysis["average_improvement"] = student_analysis[["math_improvement", "science_improvement"]].mean(axis=1)
# Calculate each student's average Post score.
student_analysis["overall_post_score"] = student_analysis[["math_post", "science_post"]].mean(axis=1)
# Convert days present into a percentage of possible attendance days.
student_analysis["attendance_rate"] = student_analysis["days_present"] / student_analysis["days_possible"] * 100
# Calculate the average length of a study session in hours.
student_analysis["hours_per_session"] = student_analysis["weekly_study_hours"] / student_analysis["weekly_sessions"]
# Round selected features for easier reading.
student_analysis[["average_improvement", "overall_post_score", "attendance_rate", "hours_per_session"]] = student_analysis[["average_improvement", "overall_post_score", "attendance_rate", "hours_per_session"]].round(1)
# Display the engineered numeric features.
display(student_analysis[["name", "average_improvement", "overall_post_score", "attendance_rate", "hours_per_session"]])

## 9. Transform numeric values into categories

Categories can aid communication, but boundaries are human decisions. Always document them.

In [ ]:
# Define ordered boundaries for attendance categories.
attendance_bins = [0, 90, 95, 100]
# Define readable labels for the attendance categories.
attendance_labels = ["Needs Attention", "Good", "Excellent"]
# Convert attendance percentages into categories.
student_analysis["attendance_category"] = pd.cut(student_analysis["attendance_rate"], bins=attendance_bins, labels=attendance_labels, include_lowest=True)
# Create a transparent rule-based support flag using two conditions.
student_analysis["support_flag"] = (student_analysis["attendance_rate"] < 90) | (student_analysis["overall_post_score"] < 80)
# Map Boolean values to student-friendly labels.
student_analysis["support_status"] = student_analysis["support_flag"].map({True: "Review", False: "On Track"})
# Display the new categorical features.
display(student_analysis[["name", "attendance_rate", "attendance_category", "overall_post_score", "support_status"]])

## 10. Summarize the finished analysis table

In [ ]:
# Group the finished table by grade and calculate named summaries.
grade_report = student_analysis.groupby("grade", as_index=False).agg(
    students=("student_id", "nunique"),
    mean_post_score=("overall_post_score", "mean"),
    mean_improvement=("average_improvement", "mean"),
    mean_attendance=("attendance_rate", "mean"),
    mean_study_hours=("weekly_study_hours", "mean"),
)
# Round the grade-level report to one decimal place.
grade_report = grade_report.round(1)
# Display the final grouped report.
display(grade_report)

## 11. Interpret without overclaiming

The combined table can reveal **patterns and associations**, but it cannot prove that attendance or study time caused higher scores. Other factors may explain the pattern, and the dataset is small.

### Exit ticket

1. When should you use `concat()` rather than `merge()`?
2. Explain the difference between an inner and left merge.
3. Why is `validate="one_to_one"` useful?
4. Name one engineered feature and explain why it may be useful.
5. Identify one limitation of the analysis.

## Skills checklist

- [ ] I can group by one or more columns.
- [ ] I can calculate named aggregations.
- [ ] I can build and flatten a pivot table.
- [ ] I can choose among concat, merge, and join.
- [ ] I check keys and row counts before and after combining data.
- [ ] I can create meaningful numeric and categorical features.
- [ ] I distinguish association from causation.